In [ ]:
! pip install jellyfish rapidfuzz

In [ ]:
import pandas as pd
import numpy as np
import re
import os
import gc
from collections import Counter, defaultdict
import jellyfish
from rapidfuzz.distance import JaroWinkler

# --- 1. Configuration & Paths ---
S1_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_source1.tsv'
S2_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_source2.tsv'
S3_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_source3.tsv'
RAW_PAIRS_PATH = '/kaggle/working/raw_pairs.tsv'
FINAL_OUTPUT_PATH = '/kaggle/working/candidate_pairs.tsv'

TOP_K_RAREST = 3
MAX_CANDIDATES = 400
WINDOW_SIZE = 6
FLUSH_EVERY = 200_000  # rows buffered in memory before writing to disk

def append_to_disk(df, path):
    if df.empty:
        return
    header = not os.path.exists(path)
    df.to_csv(path, mode='a', sep='\t', index=False, header=header)

# --- 2. Deterministic Filters ---
def clean_text(text):
    if pd.isna(text):
        return ""
    return re.sub(r'[^\w\s]', '', str(text).lower())

def has_non_latin(text):
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

def compute_metaphone(text):
    tokens = str(text).split()
    if not tokens:
        return ""
    return jellyfish.metaphone(max(tokens, key=len))


In [ ]:
# --- 3. Data Loading ---
print("Loading datasets...", flush=True)

def load_and_prep(file_path, source_name, limit=None):
    df = pd.read_csv(file_path, sep='\t', dtype=str, nrows=limit)
    df['combined_raw'] = df['business_name'].fillna('') + ' ' + df['business_address'].fillna('')
    df['clean_text'] = df['combined_raw'].apply(clean_text)
    df['clean_name'] = df['business_name'].apply(clean_text)
    
    # ADD THIS LINE: Clean the address specifically
    df['clean_address'] = df['business_address'].apply(clean_text)
    
    df['source'] = source_name
    
    # ADD 'clean_address' TO THE RETURN LIST:
    return df[['entity_id', 'clean_text', 'clean_name', 'clean_address', 'source']]

df_s1 = load_and_prep(S1_PATH, 'S1', limit=20000)
df_cand = pd.concat([load_and_prep(S2_PATH, 'S2'), load_and_prep(S3_PATH, 'S3')], ignore_index=True)

if os.path.exists(RAW_PAIRS_PATH):
    os.remove(RAW_PAIRS_PATH)



In [ ]:
# --- 4. Pass 1: Frequency Blocking (dict-based, streamed — no merge/explode) ---
print("\nPass 1: Frequency Blocking...", flush=True)

term_frequencies = Counter(
    t for text in df_cand['clean_text'] for t in text.split()
)

def get_rarest_keys(tokens):
    if not tokens:
        return ["NO_TOKEN"]
    return sorted(set(tokens), key=lambda t: term_frequencies[t])[:TOP_K_RAREST]

# Step A: provisional pass just to find which base keys are overgrown
provisional_counts = defaultdict(int)
cand_cache = []  # (entity_id, keys, name_prefix) — computed once, reused in Step B
for eid, text, name in zip(df_cand['entity_id'], df_cand['clean_text'], df_cand['clean_name']):
    keys = get_rarest_keys(text.split())
    prefix = name[:2]
    cand_cache.append((eid, keys, prefix))
    for k in keys:
        provisional_counts[k] += 1

overgrown = {k for k, count in provisional_counts.items() if count > MAX_CANDIDATES}
del provisional_counts
print(f"  {len(overgrown)} overgrown base keys will be split by name prefix", flush=True)

# Step B: build the real inverted index, splitting overgrown keys
cand_index = defaultdict(list)
for eid, keys, prefix in cand_cache:
    for k in keys:
        final_key = k + '_' + prefix if k in overgrown else k
        cand_index[final_key].append(eid)
del cand_cache
gc.collect()

# Step C: stream S1 lookups straight to disk — no intermediate joined DataFrame ever exists
buffer = []
skipped_oversized = 0
for eid, text, name in zip(df_s1['entity_id'], df_s1['clean_text'], df_s1['clean_name']):
    keys = get_rarest_keys(text.split())
    prefix = name[:2]
    for k in keys:
        final_key = k + '_' + prefix if k in overgrown else k
        bucket = cand_index.get(final_key, [])
        if len(bucket) > MAX_CANDIDATES:
            skipped_oversized += 1
            continue
        for cid in bucket:
            buffer.append((eid, cid))
    if len(buffer) >= FLUSH_EVERY:
        append_to_disk(pd.DataFrame(buffer, columns=['entity_id_x', 'entity_id_y']), RAW_PAIRS_PATH)
        buffer.clear()

if buffer:
    append_to_disk(pd.DataFrame(buffer, columns=['entity_id_x', 'entity_id_y']), RAW_PAIRS_PATH)
    buffer.clear()

print(f"  Pass 1 done. Skipped {skipped_oversized} still-oversized bucket lookups.", flush=True)
del cand_index
gc.collect()

# --- 5. Pass 2: N-Gram Shingling (dict-based, streamed) ---
print("Pass 2: N-Gram Shingling...", flush=True)

def get_ngrams(name, n=4):
    name = name.replace(" ", "")
    if len(name) < n:
        return [name] if name else []
    return [name[:n], name[-n:]]

ngram_counts = defaultdict(int)
cand_ngram_cache = []
for eid, name in zip(df_cand['entity_id'], df_cand['clean_name']):
    grams = get_ngrams(name)
    cand_ngram_cache.append((eid, grams))
    for g in grams:
        ngram_counts[g] += 1

valid_grams = {g for g, count in ngram_counts.items() if count <= MAX_CANDIDATES}
del ngram_counts

ngram_index = defaultdict(list)
for eid, grams in cand_ngram_cache:
    for g in grams:
        if g in valid_grams:
            ngram_index[g].append(eid)
del cand_ngram_cache
gc.collect()

buffer = []
for eid, name in zip(df_s1['entity_id'], df_s1['clean_name']):
    for g in get_ngrams(name):
        if g not in valid_grams:
            continue
        for cid in ngram_index.get(g, []):
            buffer.append((eid, cid))
    if len(buffer) >= FLUSH_EVERY:
        append_to_disk(pd.DataFrame(buffer, columns=['entity_id_x', 'entity_id_y']), RAW_PAIRS_PATH)
        buffer.clear()

if buffer:
    append_to_disk(pd.DataFrame(buffer, columns=['entity_id_x', 'entity_id_y']), RAW_PAIRS_PATH)
    buffer.clear()

print("  Pass 2 done.", flush=True)
del ngram_index, valid_grams
gc.collect()

# --- 6. Pass 3: Sorted Neighborhood ---
print("Pass 3: Sorted Neighborhood...", flush=True)
df_all = pd.concat([df_s1[['entity_id', 'clean_name', 'source']],
                     df_cand[['entity_id', 'clean_name', 'source']]], ignore_index=True)
df_sorted = df_all.sort_values(by='clean_name').reset_index(drop=True)[['entity_id', 'source']]
del df_all
gc.collect()

for w in range(1, WINDOW_SIZE):
    shifted = df_sorted.shift(-w)

    mask1 = (df_sorted['source'] == 'S1') & (shifted['source'].isin(['S2', 'S3']))
    mask2 = (df_sorted['source'].isin(['S2', 'S3'])) & (shifted['source'] == 'S1')
    valid = mask1 | mask2

    if valid.any():
        valid_pairs = pd.DataFrame({
            'entity_id_x': np.where(mask1[valid], df_sorted.loc[valid, 'entity_id'], shifted.loc[valid, 'entity_id']),
            'entity_id_y': np.where(mask1[valid], shifted.loc[valid, 'entity_id'], df_sorted.loc[valid, 'entity_id'])
        })
        append_to_disk(valid_pairs, RAW_PAIRS_PATH)
        del valid_pairs

    del shifted, mask1, mask2, valid
    gc.collect()

del df_sorted
gc.collect()
print("  Pass 3 done.", flush=True)




In [ ]:
def has_non_latin(text):
    return bool(re.search(r'[^\x00-\x7F]', str(text)))

# --- 7. Deduplication & Separate Text Re-Attachment ---
print("Deduplicating raw pairs...", flush=True)
all_pairs = pd.read_csv(RAW_PAIRS_PATH, sep='\t')
all_pairs = all_pairs.drop_duplicates()
all_pairs.columns = ['source1_id', 'candidate_id']
print(f"  {len(all_pairs)} unique pairs after dedup.", flush=True)

print("Re-attaching Name and Address for filtering...", flush=True)
all_pairs['name_s1'] = all_pairs['source1_id'].map(df_s1.set_index('entity_id')['clean_name']).fillna('')
all_pairs['name_cand'] = all_pairs['candidate_id'].map(df_cand.set_index('entity_id')['clean_name']).fillna('')

all_pairs['addr_s1'] = all_pairs['source1_id'].map(df_s1.set_index('entity_id')['clean_address']).fillna('')
all_pairs['addr_cand'] = all_pairs['candidate_id'].map(df_cand.set_index('entity_id')['clean_address']).fillna('')

# Clean up memory immediately
#del df_s1, df_cand
#gc.collect()

# --- 8. Smart Filtering (VIP Pass + Address Rescue) ---
print(f"Evaluating {len(all_pairs)} unique pairs...", flush=True)

# Pre-allocate a fast boolean array (defaults to False)
mask_jw = np.zeros(len(all_pairs), dtype=bool)

for i, (n1, n2, a1, a2) in enumerate(zip(all_pairs['name_s1'], all_pairs['name_cand'], all_pairs['addr_s1'], all_pairs['addr_cand'])):
    
    # 1. VIP ACCESS: If ANY regional text is detected, skip Jaro-Winkler and pass it
    if has_non_latin(n1) or has_non_latin(n2):
        mask_jw[i] = True
        continue
        
    # 2. NAME CHECK: Standard English check at 0.70
    if JaroWinkler.normalized_similarity(n1, n2) >= 0.70:
        mask_jw[i] = True
        continue
        
    # 3. ADDRESS RESCUE: If Name fails (acronym/length trap), check if Address is a hard match
    if len(a1) > 5 and len(a2) > 5:
        if JaroWinkler.normalized_similarity(a1, a2) >= 0.65:
            mask_jw[i] = True

final_df = all_pairs[mask_jw][['source1_id', 'candidate_id']]
final_df.to_csv(FINAL_OUTPUT_PATH, sep='\t', index=False)
print(f"Complete! Generated {len(final_df)} filtered pairs.")

In [ ]:
import pandas as pd

GT_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_ground_truth.tsv'
S1_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_source1.tsv'
RAW_PATH = '/kaggle/working/raw_pairs.tsv'
CANDS_PATH = '/kaggle/working/candidate_pairs.tsv'
PROCESSED_S1_RECORDS = 20000

print("Loading exact S1 IDs processed...", flush=True)
# 1. Grab the EXACT 20,000 IDs we pushed through the pipeline
s1_processed = pd.read_csv(S1_PATH, sep='\t', usecols=['entity_id'], nrows=PROCESSED_S1_RECORDS)
valid_s1_ids = set(s1_processed['entity_id'])

print("Loading Ground Truth...", flush=True)
# 2. Filter Ground Truth to ONLY grade the S1 IDs we processed
gt = pd.read_csv(GT_PATH, sep='\t')
gt_subset = gt[gt['source1_entity_id'].isin(valid_s1_ids)].copy()

true_pairs = set()
for _, row in gt_subset.iterrows():
    s1_id = row['source1_entity_id']
    if pd.notna(row['matched_entity_ids']):
        matches = [m.strip() for m in str(row['matched_entity_ids']).split(',') if m.strip()]
        for m in matches:
            true_pairs.add((s1_id, m))

total_true = len(true_pairs)
print(f"Total True Pairs to find for this batch: {total_true}")

# 3. Check how many survived the Blocking Passes (Raw)
raw = pd.read_csv(RAW_PATH, sep='\t')
raw_set = set(zip(raw['entity_id_x'], raw['entity_id_y']))
raw_found = len(true_pairs.intersection(raw_set))
print(f"Recall after Blocking (Raw Pairs): {(raw_found / total_true) * 100:.2f}%")

# 4. Check how many survived Jaro-Winkler (Candidates)
cands = pd.read_csv(CANDS_PATH, sep='\t')
cands_set = set(zip(cands['source1_id'], cands['candidate_id']))
cands_found = len(true_pairs.intersection(cands_set))
print(f"Recall after Jaro-Winkler (Candidates): {(cands_found / total_true) * 100:.2f}%")

In [ ]:
import pandas as pd

GT_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_ground_truth.tsv'
CANDS_PATH = '/kaggle/working/candidate_pairs.tsv'

# --- THE FIX ---
# Set this to match the 'limit' you used in Step 3. 
# (If you do a full run later, you can just change this to len(gt))
PROCESSED_S1_RECORDS = 20000 

# 1. Average true matches per S1 record in the Ground Truth
gt = pd.read_csv(GT_PATH, sep='\t')

# Match the ground truth to the exact batch you processed
gt_subset = gt.head(PROCESSED_S1_RECORDS).copy() 
gt_subset['match_count'] = gt_subset['matched_entity_ids'].fillna('').apply(
    lambda x: len([i for i in str(x).split(',') if i.strip()])
)

print(f"S1 Records Processed: {PROCESSED_S1_RECORDS}")
print(f"Average True Matches per S1 (Ground Truth): {gt_subset['match_count'].mean():.2f}")

# 2. Average candidates generated per S1 record
cands = pd.read_csv(CANDS_PATH, sep='\t')
total_candidates = len(cands)

# Divide the candidates by the EXACT number of records we fed into the pipeline
true_average_candidates = total_candidates / PROCESSED_S1_RECORDS

print(f"Total Candidates Generated: {total_candidates}")
print(f"Average Candidates per S1 (Pipeline Output): {true_average_candidates:.2f}")

In [ ]:
import pandas as pd

GT_PATH = '/kaggle/input/datasets/renganathc/dataset/dataset/train/train_ground_truth.tsv'
CANDS_PATH = '/kaggle/working/raw_pairs.tsv'

# --- THE FIX ---
# Set this to match the 'limit' you used in Step 3. 
# (If you do a full run later, you can just change this to len(gt))
PROCESSED_S1_RECORDS = 20000 

# 1. Average true matches per S1 record in the Ground Truth
gt = pd.read_csv(GT_PATH, sep='\t')

# Match the ground truth to the exact batch you processed
gt_subset = gt.head(PROCESSED_S1_RECORDS).copy() 
gt_subset['match_count'] = gt_subset['matched_entity_ids'].fillna('').apply(
    lambda x: len([i for i in str(x).split(',') if i.strip()])
)

print(f"S1 Records Processed: {PROCESSED_S1_RECORDS}")
print(f"Average True Matches per S1 (Ground Truth): {gt_subset['match_count'].mean():.2f}")

# 2. Average candidates generated per S1 record
cands = pd.read_csv(CANDS_PATH, sep='\t')
total_candidates = len(cands)

# Divide the candidates by the EXACT number of records we fed into the pipeline
true_average_candidates = total_candidates / PROCESSED_S1_RECORDS

print(f"Total Candidates Generated: {total_candidates}")
print(f"Average Candidates per S1 (Pipeline Output): {true_average_candidates:.2f}")